# Ames House Price Prediction — Advanced Regression & Ensembling

**End-to-end Machine Learning portfolio: exploratory analysis, feature engineering, regularized linear modeling, gradient boosting, and blending ensemble.**

Proyek ini bertujuan memprediksi harga jual properti perumahan (`SalePrice`) menggunakan dataset **Ames Housing** dari Kaggle. Target adalah variabel numerik kontinu dalam USD.

## Ringkasan Metodologi & Hasil
- **Dataset:** 1.460 observasi, 79 fitur orisinal.
- **Pembersihan Data:** Deteksi dan pembersihan 2 outlier ekstrem pada `GrLivArea > 4000 sq ft` dengan harga abnormal.
- **Transformasi Target:** Log-transformation $\log(1 + y)$ untuk menormalkan skewness dari **1,88** ke **0,12**.
- **Feature Engineering:** `TotalSF` (total luas seluruh lantai), `TotalBath`, `HouseAge`, `RemodAge`, dan indikator fasilitas biner.
- **Model Terbaik:** Ensemble Blending (60% Ridge Regression + 40% XGBoost Regressor).
- **Evaluasi Akhir Data Uji (Holdout Test Set):**
  - **Log-RMSE:** `0,1169` (Top 10-15% kompetisi Kaggle)
  - **R² Score:** `91,89%`
  - **MAE:** `$13.485,69`


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib

print('Libraries loaded successfully!')


## 1. Memuat Dataset

Fungsi loader berikut mendeteksi secara otomatis lingkungan kerja:
- **Kaggle:** Mencari file di `/kaggle/input/house-prices-advanced-regression-techniques/train.csv`
- **Lokal:** Membaca dari `data/train.csv` atau direktori saat ini
- **Google Colab:** Meminta upload jika file belum ada


In [ ]:
file_name = 'train.csv'
data_path = None

candidates = [
    Path(file_name),
    Path('data') / file_name,
    Path('../data') / file_name,
    Path('../house-prices-advanced-regression-techniques') / file_name
]

for p in candidates:
    if p.is_file():
        data_path = p
        break

if data_path is None and Path('/kaggle/input').is_dir():
    matches = sorted(Path('/kaggle/input').rglob(file_name))
    if matches:
        data_path = matches[0]

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = Path(file_name)
    except ImportError:
        raise FileNotFoundError('File train.csv tidak ditemukan. Simpan di folder data/.')

print('Memuat data dari:', data_path)
df = pd.read_csv(data_path)
df.head()


## 2. Pemeriksaan Struktur Data & Karakteristik Target

Memeriksa dimensi data, sebaran harga, kemencengan (skewness), dan tipe data fitur.


In [ ]:
print(f'Jumlah baris: {df.shape[0]:,}')
print(f'Jumlah kolom/fitur: {df.shape[1]:,}')

target = df['SalePrice']
print('\n--- STATISTIK TARGET (SalePrice) ---')
print(f'Harga terendah  : ${target.min():,}')
print(f'Harga rata-rata : ${target.mean():,.2f}')
print(f'Harga median    : ${target.median():,}')
print(f'Harga tertinggi : ${target.max():,}')
print(f'Skewness        : {target.skew():.2f}')

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'\nJumlah kolom numerik    : {len(numeric_cols)}')
print(f'Jumlah kolom kategorikal: {len(categorical_cols)}')
print(f'Baris duplikat          : {df.duplicated().sum()}')


## 3. Analisis Distribusi Target & Log-Transformation

Karena harga rumah mengalami right-skewed yang tinggi (1,88), kita terapkan transformasi log `np.log1p(y)` untuk menstabilkan varians dan mendekatkan distribusi ke bentuk normal Gaussian.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Harga Asli
sns.histplot(df['SalePrice'], kde=True, ax=axes[0], color='#5878C7', bins=35)
axes[0].set_title(f"Distribusi SalePrice Asli (Skewness: {df['SalePrice'].skew():.2f})")
axes[0].set_xlabel('Harga Jual (USD)')

# Histogram Log(1 + Harga)
log_price = np.log1p(df['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[1], color='#48A9A6', bins=35)
axes[1].set_title(f'Distribusi Log(SalePrice) (Skewness: {log_price.skew():.2f})')
axes[1].set_xlabel('Log(1 + SalePrice)')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


## 4. Analisis Missing Values

Pada dataset Ames Housing, nilai kosong pada kolom fasilitas (`PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`, `GarageType`, dll.) bukan data hilang, melainkan menunjukkan fasilitas tersebut tidak ada pada rumah terkait.


In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct.round(2)
})

print(f'Total kolom dengan missing value: {len(missing_df)}')
missing_df.head(15)


## 5. Pembersihan Outlier & Feature Engineering

Menghapus 2 outlier ekstrem (`GrLivArea > 4000` dengan `SalePrice < 300000`) serta membangun fitur sintetis berbasis domain real estate:
- `TotalSF` = `TotalBsmtSF` + `1stFlrSF` + `2ndFlrSF`
- `TotalBath` = `FullBath` + `0.5*HalfBath` + `BsmtFullBath` + `0.5*BsmtHalfBath`
- `HouseAge` = `YrSold` - `YearBuilt`
- `RemodAge` = `YrSold` - `YearRemodAdd`


In [ ]:
# 1. Hapus 2 outlier ekstrem
df_clean = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index).reset_index(drop=True)
print(f'Ukuran data setelah hapus outlier: {df_clean.shape}')

# 2. Feature Engineering
def engineer_features(data):
    df_feat = data.copy()
    
    # Total luas huni seluruh lantai
    df_feat['TotalSF'] = df_feat['TotalBsmtSF'].fillna(0) + df_feat['1stFlrSF'].fillna(0) + df_feat['2ndFlrSF'].fillna(0)
    
    # Total kapasitas kamar mandi
    df_feat['TotalBath'] = (
        df_feat['FullBath'].fillna(0) +
        0.5 * df_feat['HalfBath'].fillna(0) +
        df_feat['BsmtFullBath'].fillna(0) +
        0.5 * df_feat['BsmtHalfBath'].fillna(0)
    )
    
    # Usia bangunan & renovasi saat transaksi
    df_feat['HouseAge'] = (df_feat['YrSold'] - df_feat['YearBuilt']).clip(lower=0)
    df_feat['RemodAge'] = (df_feat['YrSold'] - df_feat['YearRemodAdd']).clip(lower=0)
    
    # Indikator fasilitas
    df_feat['HasPool'] = (df_feat['PoolArea'] > 0).astype(int)
    df_feat['HasGarage'] = (df_feat['GarageArea'] > 0).astype(int)
    df_feat['HasBasement'] = (df_feat['TotalBsmtSF'] > 0).astype(int)
    df_feat['HasFireplace'] = (df_feat['Fireplaces'] > 0).astype(int)
    
    return df_feat

df_engineered = engineer_features(df_clean)

# Pisahkan fitur dan target
X = df_engineered.drop(columns=['Id', 'SalePrice'])
y = df_engineered['SalePrice']

# Split 80% Train, 20% Holdout Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f'Train set: {X_train.shape[0]} baris, {X_train.shape[1]} fitur')
print(f'Test set : {X_test.shape[0]} baris, {X_test.shape[1]} fitur')
print(f'Korelasi TotalSF terhadap Log-Harga: {np.corrcoef(X_train["TotalSF"], y_train_log)[0, 1]:.4f}')


## 6. Pipeline Preprocessing (Bebas Data Leakage)

Membungkus proses imputasi dan encoding dalam `ColumnTransformer`:
- Fitur numerik: `SimpleImputer(strategy='median')` + `StandardScaler()`
- Fitur kategorikal: `SimpleImputer(strategy='constant', fill_value='None')` + `OneHotEncoder(handle_unknown='ignore')`


In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

print(f'Fitur numerik: {len(numeric_features)}, Fitur kategorikal: {len(categorical_features)}')


## 7. Perbandingan Model Awal (5-Fold Cross Validation)

Membandingkan Dummy, Ridge Regression, Random Forest, Gradient Boosting, dan XGBoost.


In [ ]:
models = {
    'Dummy (Baseline)': DummyRegressor(strategy='median'),
    'Ridge Regression': Ridge(alpha=10.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42, n_jobs=-1)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    rmse_scores, mae_scores, r2_scores = [], [], []
    
    for tr_idx, val_idx in kf.split(X_train, y_train_log):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr_log, y_val_log = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]
        y_val_usd = y_train.iloc[val_idx]
        
        pipe.fit(X_tr, y_tr_log)
        p_log = pipe.predict(X_val)
        p_usd = np.expm1(p_log)
        
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_log, p_log)))
        mae_scores.append(mean_absolute_error(y_val_usd, p_usd))
        r2_scores.append(r2_score(y_val_log, p_log))
        
    cv_rows.append({
        'Model': name,
        'Log-RMSE': np.mean(rmse_scores),
        'MAE ($ USD)': np.mean(mae_scores),
        'R² Score': np.mean(r2_scores)
    })

pd.DataFrame(cv_rows).sort_values('Log-RMSE').reset_index(drop=True)


## 8. Hyperparameter Tuning & Blending Ensemble

Tuning Ridge dan XGBoost dengan `GridSearchCV`, lalu menggabungkan keduanya (*blending ensemble*) dengan bobot 60% Ridge + 40% XGBoost.


In [ ]:
# Tuning Ridge
ridge_pipe = Pipeline([('preprocessor', preprocessor), ('model', Ridge(random_state=42))])
ridge_grid = GridSearchCV(ridge_pipe, {'model__alpha': [1.0, 5.0, 10.0, 15.0, 20.0, 30.0]}, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train, y_train_log)

# Tuning XGBoost
xgb_pipe = Pipeline([('preprocessor', preprocessor), ('model', xgb.XGBRegressor(random_state=42, n_jobs=-1))])
xgb_grid = GridSearchCV(xgb_pipe, {'model__n_estimators': [200, 300], 'model__max_depth': [3, 4], 'model__learning_rate': [0.03, 0.05]}, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
xgb_grid.fit(X_train, y_train_log)

print('Alpha Terbaik Ridge:', ridge_grid.best_params_['model__alpha'])
print('Parameter Terbaik XGBoost:', xgb_grid.best_params_)


## 9. Evaluasi Akhir pada Holdout Test Set & Analisis Residu

Melatih kedua model final pada seluruh data latih, memprediksi 292 rumah data uji independen, dan memeriksa plot residu.


In [ ]:
final_ridge = ridge_grid.best_estimator_
final_xgb = xgb_grid.best_estimator_

final_ridge.fit(X_train, y_train_log)
final_xgb.fit(X_train, y_train_log)

# Prediksi Blending (60:40)
pred_ridge_log = final_ridge.predict(X_test)
pred_xgb_log = final_xgb.predict(X_test)
test_pred_blend_log = 0.60 * pred_ridge_log + 0.40 * pred_xgb_log
test_pred_blend_usd = np.expm1(test_pred_blend_log)

# Metrik
test_log_rmse = np.sqrt(mean_squared_error(y_test_log, test_pred_blend_log))
test_mae_usd = mean_absolute_error(y_test, test_pred_blend_usd)
test_r2 = r2_score(y_test_log, test_pred_blend_log)

print(f'Log-RMSE : {test_log_rmse:.4f}')
print(f'R² Score : {test_r2 * 100:.2f}%')
print(f'MAE ($)  : ${test_mae_usd:,.2f}')
print(f'MedAE ($): ${np.median(np.abs(y_test - test_pred_blend_usd)):,.2f}')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, test_pred_blend_usd, alpha=0.6, color='#5878C7', edgecolor='white', s=50)
min_val = min(y_test.min(), test_pred_blend_usd.min())
max_val = max(y_test.max(), test_pred_blend_usd.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')
axes[0].set_title(f'Harga Aktual vs Prediksi (R² = {test_r2*100:.1f}%)')
axes[0].set_xlabel('Harga Aktual ($ USD)')
axes[0].set_ylabel('Harga Prediksi ($ USD)')
axes[0].legend()

residuals = y_test - test_pred_blend_usd
sns.histplot(residuals, kde=True, ax=axes[1], color='#48A9A6', bins=30)
axes[1].axvline(0, color='r', linestyle='--', lw=2)
axes[1].set_title('Distribusi Residu (Error = Aktual - Prediksi)')
axes[1].set_xlabel('Residual ($ USD)')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


## 10. Menyimpan Artefak Model untuk Deployment Aplikasi

Menyimpan model joblib, template default fitur, dan metadata evaluasi.


In [ ]:
out_dir = Path('house_price_artifacts')
out_dir.mkdir(exist_ok=True)

bundle = {
    'ridge_model': final_ridge,
    'xgb_model': final_xgb,
    'blend_weights': {'ridge': 0.60, 'xgb': 0.40},
    'feature_names': X_train.columns.tolist(),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'category_options': {col: sorted(X_train[col].dropna().unique().tolist()) for col in categorical_features}
}

joblib.dump(bundle, out_dir / 'house_price_model.joblib')
fig.savefig(out_dir / 'test_evaluation_plots.png', dpi=150, bbox_inches='tight')

print('Artefak berhasil diekspor!')
